# YOLO License Plate Detector Training

Notebook ini melatih YOLO untuk menemukan **lokasi plat nomor**. Model ini tidak membaca karakter; crop hasil deteksi diteruskan ke OCR/recognizer dari notebook `01_license_plate_detection_and_ocr_baseline.ipynb`.

Pipeline: **Pascal VOC XML -> audit -> fixed split -> YOLO dataset -> train -> validate -> predict -> export**.

> Dataset saat ini banyak berisi crop plat yang memenuhi hampir seluruh frame. Model baseline tetap dapat dilatih, tetapi generalisasi ke IP Camera harus diuji dan ditingkatkan menggunakan foto kendaraan utuh dari sudut kamera aktual.

## 0. Environment
Pilih kernel `Python (.venv) - Smart Parking System`. Training GPU memerlukan PyTorch CUDA; environment saat ini otomatis menggunakan CPU bila CUDA tidak tersedia.

In [ ]:
import csv
import hashlib
import random
import shutil
import statistics
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
from IPython.display import Image as DisplayImage, display
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration and paths

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'apps' / 'vision').is_dir() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Smart Parking System project root was not found.')


PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'indonesian-plate-number' / 'plate_number_recognition_datasets_v1'
SOURCE_IMAGE_ROOT = SOURCE_ROOT / 'images'
SOURCE_ANNOTATION_ROOT = SOURCE_ROOT / 'annotations'
YOLO_DATASET_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'yolo_license_plate'
DATASET_YAML = YOLO_DATASET_ROOT / 'dataset.yaml'
MANIFEST_CSV = YOLO_DATASET_ROOT / 'manifest.csv'
BASE_MODEL_PATH = PROJECT_ROOT / 'ml' / 'weights' / 'yolo11n.pt'
EXPERIMENT_ROOT = PROJECT_ROOT / 'ml' / 'experiments' / 'yolo_license_plate_detector'
RUN_NAME = 'yolo11n_plate_v1'

SEED = 42
VALIDATION_RATIO = 0.20
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 8
PATIENCE = 15
TRAINING_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUN_TRAINING = True  # Ubah True hanya saat siap memulai training.
RUN_EXPORT = False

if not SOURCE_IMAGE_ROOT.is_dir() or not SOURCE_ANNOTATION_ROOT.is_dir():
    raise FileNotFoundError(f'Source dataset is incomplete: {SOURCE_ROOT}')
if not BASE_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Pretrained model is missing: {BASE_MODEL_PATH}')

print(f'Source: {SOURCE_ROOT}')
print(f'Prepared dataset: {YOLO_DATASET_ROOT}')
print(f'Device: {TRAINING_DEVICE}')
print(f'RUN_TRAINING: {RUN_TRAINING}')

## 2. Parse and audit Pascal VOC annotations
Bounding box divalidasi terhadap ukuran gambar. `scene` berarti plat tidak mendominasi frame; `plate_crop` berarti setidaknya satu box memakai lebih dari 50% area frame.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def parse_voc_annotation(xml_path: Path) -> dict:
    root = ET.parse(xml_path).getroot()
    width = int(float(root.findtext('size/width', '0')))
    height = int(float(root.findtext('size/height', '0')))
    boxes = []
    for obj in root.findall('object'):
        if (obj.findtext('name') or '').strip().lower() != 'plate':
            continue
        bbox = obj.find('bndbox')
        if bbox is None:
            continue
        x1, y1, x2, y2 = [float(bbox.findtext(key, '0')) for key in ('xmin', 'ymin', 'xmax', 'ymax')]
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(float(width), x2), min(float(height), y2)
        if width > 0 and height > 0 and x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return {'width': width, 'height': height, 'boxes': boxes}


def source_records(split: str) -> list[dict]:
    records = []
    for image_path in sorted((SOURCE_IMAGE_ROOT / split).iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        xml_path = SOURCE_ANNOTATION_ROOT / split / f'{image_path.stem}.xml'
        if not xml_path.is_file():
            raise FileNotFoundError(f'Missing annotation: {xml_path}')
        annotation = parse_voc_annotation(xml_path)
        if not annotation['boxes']:
            raise ValueError(f'No valid plate boxes: {xml_path}')
        area_ratios = [((x2 - x1) * (y2 - y1)) / (annotation['width'] * annotation['height']) for x1, y1, x2, y2 in annotation['boxes']]
        records.append({
            'source_split': split, 'image_path': image_path, 'xml_path': xml_path,
            'width': annotation['width'], 'height': annotation['height'], 'boxes': annotation['boxes'],
            'max_area_ratio': max(area_ratios), 'scene_type': 'plate_crop' if max(area_ratios) > 0.50 else 'scene',
        })
    return records


train_source_records = source_records('train')
test_source_records = source_records('test')
all_source_records = train_source_records + test_source_records
all_ratios = [record['max_area_ratio'] for record in all_source_records]
print(f'Images: {len(all_source_records):,}')
print(f'Boxes: {sum(len(record["boxes"]) for record in all_source_records):,}')
print(f'Scene types: {dict(Counter(record["scene_type"] for record in all_source_records))}')
print(f'Median maximum box area: {statistics.median(all_ratios):.2%}')

## 3. Create deterministic stratified split
Folder Kaggle `test` dipertahankan sebagai final test. Folder `train` dibagi menjadi train/validation secara deterministik dan distratifikasi berdasarkan `scene_type`.

In [ ]:
def stratified_train_val_split(records: list[dict], validation_ratio: float, seed: int) -> tuple[list[dict], list[dict]]:
    rng = random.Random(seed)
    grouped = {}
    for record in records:
        grouped.setdefault(record['scene_type'], []).append(record)
    train_records, val_records = [], []
    for group in grouped.values():
        group = group.copy()
        rng.shuffle(group)
        val_count = max(1, round(len(group) * validation_ratio))
        val_records.extend(group[:val_count])
        train_records.extend(group[val_count:])
    rng.shuffle(train_records)
    rng.shuffle(val_records)
    return train_records, val_records


train_records, val_records = stratified_train_val_split(train_source_records, VALIDATION_RATIO, SEED)
split_records = {'train': train_records, 'val': val_records, 'test': test_source_records}
for split, records_for_split in split_records.items():
    distribution = Counter(record['scene_type'] for record in records_for_split)
    print(f'{split:5s}: {len(records_for_split):3d} images | {dict(distribution)}')

content_hash_splits = {}
for split, records_for_split in split_records.items():
    for record in records_for_split:
        digest = hashlib.sha256(record['image_path'].read_bytes()).hexdigest()
        content_hash_splits.setdefault(digest, set()).add(split)
cross_split_duplicates = [splits for splits in content_hash_splits.values() if len(splits) > 1]
if cross_split_duplicates:
    raise RuntimeError(f'Found {len(cross_split_duplicates)} duplicate image group(s) across splits.')
print('Cross-split duplicate images: 0')

## 4. Convert dataset to YOLO format
Dataset hasil konversi berada di `data/processed/yolo_license_plate`. Manifest mempertahankan hubungan ke file sumber. Menjalankan ulang cell ini akan mengganti hanya folder prepared dataset tersebut.

In [ ]:
def voc_box_to_yolo(box: tuple[float, float, float, float], width: int, height: int) -> tuple[float, float, float, float]:
    x1, y1, x2, y2 = box
    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)


def safe_sample_name(record: dict, split: str, index: int) -> str:
    digest = hashlib.sha1(str(record['image_path']).encode('utf-8')).hexdigest()[:8]
    return f'{split}_{index:04d}_{digest}'


if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)

manifest_rows = []
for split, records_for_split in split_records.items():
    image_output_dir = YOLO_DATASET_ROOT / 'images' / split
    label_output_dir = YOLO_DATASET_ROOT / 'labels' / split
    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)
    for index, record in enumerate(records_for_split, start=1):
        sample_name = safe_sample_name(record, split, index)
        target_image = image_output_dir / f'{sample_name}{record["image_path"].suffix.lower()}'
        target_label = label_output_dir / f'{sample_name}.txt'
        shutil.copy2(record['image_path'], target_image)
        label_lines = []
        for box in record['boxes']:
            x_center, y_center, box_width, box_height = voc_box_to_yolo(box, record['width'], record['height'])
            label_lines.append(f'0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}')
        target_label.write_text('\n'.join(label_lines) + '\n', encoding='ascii')
        manifest_rows.append({
            'split': split, 'prepared_image': str(target_image.relative_to(YOLO_DATASET_ROOT)),
            'source_image': str(record['image_path'].relative_to(PROJECT_ROOT)),
            'scene_type': record['scene_type'], 'box_count': len(record['boxes']),
        })

dataset_config = {
    'path': str(YOLO_DATASET_ROOT.resolve()), 'train': 'images/train', 'val': 'images/val',
    'test': 'images/test', 'names': {0: 'license_plate'},
}
DATASET_YAML.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding='utf-8')
with MANIFEST_CSV.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=manifest_rows[0].keys())
    writer.writeheader()
    writer.writerows(manifest_rows)
print(f'Prepared {len(manifest_rows):,} images')
print(f'Dataset YAML: {DATASET_YAML}')
print(f'Manifest: {MANIFEST_CSV}')

## 5. Validate and visualize converted labels

In [ ]:
def draw_yolo_labels(image_path: Path, label_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    height, width = image.shape[:2]
    for line in label_path.read_text(encoding='ascii').splitlines():
        class_id, x_center, y_center, box_width, box_height = map(float, line.split())
        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)
        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 220, 0), max(2, width // 300))
    return image


for split in ('train', 'val', 'test'):
    image_paths = sorted((YOLO_DATASET_ROOT / 'images' / split).iterdir())
    label_paths = sorted((YOLO_DATASET_ROOT / 'labels' / split).glob('*.txt'))
    if len(image_paths) != len(label_paths):
        raise RuntimeError(f'Image/label count mismatch in {split}')
    print(f'{split}: {len(image_paths)} image-label pairs')

preview_paths = random.Random(SEED).sample(sorted((YOLO_DATASET_ROOT / 'images' / 'val').iterdir()), k=min(6, len(val_records)))
for image_path in preview_paths:
    annotated = draw_yolo_labels(image_path, YOLO_DATASET_ROOT / 'labels' / 'val' / f'{image_path.stem}.txt')
    ok, encoded = cv2.imencode('.jpg', annotated)
    print(image_path.name)
    if ok:
        display(DisplayImage(data=encoded.tobytes(), width=700))

## 6. Train YOLO plate detector
Ubah `RUN_TRAINING = True` pada configuration cell, lalu jalankan cell ini. Mulai dari YOLO11n agar eksperimen cepat. Untuk training cloud/GPU, salin prepared dataset dan notebook tanpa memasukkan output model ke Git.

In [ ]:
training_result = None
if RUN_TRAINING:
    model = YOLO(str(BASE_MODEL_PATH))
    training_result = model.train(
        data=str(DATASET_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=TRAINING_DEVICE, workers=0, seed=SEED, deterministic=True, patience=PATIENCE,
        project=str(EXPERIMENT_ROOT), name=RUN_NAME, exist_ok=True, pretrained=True,
        cache=False, close_mosaic=10, plots=True, verbose=True,
    )
else:
    print('Training skipped. Set RUN_TRAINING=True in the configuration cell when ready.')

## 7. Validate best checkpoint
Validation memakai split `val`; final evaluation memakai split `test` yang tidak digunakan untuk memilih checkpoint.

In [ ]:
BEST_MODEL_PATH = EXPERIMENT_ROOT / RUN_NAME / 'weights' / 'best.pt'
if BEST_MODEL_PATH.is_file():
    trained_model = YOLO(str(BEST_MODEL_PATH))
    validation_metrics = trained_model.val(data=str(DATASET_YAML), split='val', imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, workers=0, plots=True)
    test_metrics = trained_model.val(data=str(DATASET_YAML), split='test', imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, workers=0, plots=True)
    print(f'Validation mAP50: {validation_metrics.box.map50:.4f}')
    print(f'Validation mAP50-95: {validation_metrics.box.map:.4f}')
    print(f'Test mAP50: {test_metrics.box.map50:.4f}')
    print(f'Test mAP50-95: {test_metrics.box.map:.4f}')
else:
    trained_model = None
    print(f'No trained checkpoint yet: {BEST_MODEL_PATH}')

## 8. Inspect predictions on held-out test images

In [ ]:
if trained_model is not None:
    test_images = sorted((YOLO_DATASET_ROOT / 'images' / 'test').iterdir())[:12]
    predictions = trained_model.predict(test_images, conf=0.25, imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, verbose=False)
    for prediction in predictions:
        annotated = prediction.plot()
        ok, encoded = cv2.imencode('.jpg', annotated)
        print(f'{Path(prediction.path).name}: {len(prediction.boxes)} detection(s)')
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=800))
else:
    print('Train the model first to inspect predictions.')

## 9. Upload a vehicle image for detector testing
Upload gambar kendaraan utuh setelah checkpoint tersedia. Cell menampilkan bounding box dan confidence; OCR belum dijalankan di tahap ini.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

detector_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Choose image')
detector_button = widgets.Button(description='Detect plate', button_style='primary', icon='search')
detector_output = widgets.Output()


def first_uploaded_file(value):
    if isinstance(value, dict):
        return next(iter(value.values()), None)
    return value[0] if value else None


def detect_uploaded_image(_):
    with detector_output:
        clear_output(wait=True)
        if trained_model is None:
            print(f'Trained model not found: {BEST_MODEL_PATH}')
            return
        payload = first_uploaded_file(detector_upload.value)
        if payload is None:
            print('Choose an image first.')
            return
        image = cv2.imdecode(np.frombuffer(bytes(payload['content']), dtype=np.uint8), cv2.IMREAD_COLOR)
        if image is None:
            print('The uploaded file is not a readable image.')
            return
        prediction = trained_model.predict(image, conf=0.25, imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, verbose=False)[0]
        annotated = prediction.plot()
        ok, encoded = cv2.imencode('.jpg', annotated)
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=900))
        print(f'Detections: {len(prediction.boxes)}')
        for index, box in enumerate(prediction.boxes, start=1):
            print(f'Plate {index}: confidence={float(box.conf[0]):.2%}, xyxy={[round(value, 1) for value in box.xyxy[0].tolist()]}')


detector_button.on_click(detect_uploaded_image)
display(widgets.VBox([detector_upload, detector_button, detector_output]))

## 10. Export deployment model
ONNX cocok sebagai baseline portable untuk cloud/CPU inference. Export hanya dilakukan saat `RUN_EXPORT = True`.

In [ ]:
if RUN_EXPORT:
    if trained_model is None:
        raise FileNotFoundError(f'Train a checkpoint before export: {BEST_MODEL_PATH}')
    export_path = trained_model.export(format='onnx', imgsz=IMAGE_SIZE, dynamic=True, simplify=True)
    print(f'Exported model: {export_path}')
else:
    print('Export skipped. Set RUN_EXPORT=True when the trained checkpoint is ready.')

## 11. Acceptance criteria and next data iteration

Model belum dianggap siap hanya karena mAP tinggi pada dataset ini. Minimal evaluasi berikut diperlukan:

- mAP50 dan recall pada held-out test.
- Recall terpisah untuk mobil, motor, siang, malam, glare, blur, dan jarak berbeda.
- Ukuran plat dalam piksel pada capture zone aktual.
- False positive pada kendaraan tanpa plat terlihat dan objek mirip plat.
- End-to-end exact-match setelah crop YOLO diteruskan ke recognizer.

Prioritas data berikutnya adalah foto kendaraan utuh dari posisi IP Camera yang direncanakan. Tambahkan hard negatives dan variasi kondisi nyata, lalu train versi baru tanpa mengubah final test set.